In [ ]:
# Resolve configurable_mdp root for common working directories
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    module_root = cwd
elif (cwd / "configurable_mdp" / "src").exists():
    module_root = cwd / "configurable_mdp"
elif cwd.name == "notebooks" and (cwd.parent / "src").exists():
    module_root = cwd.parent
else:
    raise RuntimeError(f"Could not locate configurable_mdp root from cwd={cwd}")

module_path = str(module_root)
if module_path not in sys.path:
    sys.path.insert(0, module_path)

print(f"module_path = {module_path}")
print(f"src import ready: {(Path(module_path) / 'src').exists()}")

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import orbax

# JAX: force CPU for this notebook kernel session only
os.environ["JAX_PLATFORMS"] = "cpu"
import jax
jax.config.update("jax_platform_name", "cpu")

import jax.numpy as jnp
from src.algorithms.value_iteration_and_prediction import general_value_iteration
from train_four_rooms_hpgd import environment_setup
from visualization_functions import plot_UL_rewards, plot_incentive_grid

# Parameters for plotting
colors = plt.cm.Dark2.colors
linewidth = 2.5
figsize = (8.0, 4.8)  # slightly larger graph box while keeping readability
base_font_size = 20
legend_font_size = 18  # small-caps labels look larger, so keep legend slightly smaller
axis_label_font_size = 22  # make axis labels slightly larger
plt.rcParams.update(
    {
        'font.size': base_font_size,
        'legend.fontsize': legend_font_size,
        'axes.labelsize': axis_label_font_size,
        'text.usetex': True,
        'axes.linewidth': linewidth,
        'xtick.major.width': linewidth,
        'ytick.major.width': linewidth,
        'xtick.major.size': 2*linewidth,
        'ytick.major.size': 2*linewidth,
        'axes.prop_cycle': plt.cycler(color=plt.cm.Dark2.colors),
        "lines.linewidth": linewidth,
    }
)

# Load the data

In [ ]:
save_figs = True

data_dir = "../data/experiment_reg_lambda_0_001_total_steps_100"
# save_dir = os.path.join(data_dir, "figures")
save_dir = os.getcwd()

algorithms = ["bchg", "hpgd", "hpgd_sarsa", "sobirl", "baseline", "hpgd_oracle"]

# Note: the order in the legend follows the order in list "algorithms"

# Plot settings
rolling_window = 100
plot_top = 10
seed_idx = None
legend_names = {
    "bchg": r"\textsc{BC-HG}",
    "hpgd": r"\textsc{HPGD (MC)}",
    "hpgd_sarsa": r"\textsc{HPGD (SA)}",
    "hpgd_oracle": r"\textsc{HPGD (oracle)}",
    "sobirl": r"\textsc{SoBiRL}",
    "baseline": r"\textsc{Naive-PGD}",
}
line_styles = {
    "bchg": "-",
    "hpgd": "--",
    "hpgd_sarsa": "-.",
    "hpgd_oracle": ":",
    "sobirl": (0, (2, 4, 2, 4)),
    "baseline": (0, (1, 3)),
}
algo_colors = {
    "bchg": colors[2],
    "hpgd": colors[1],
    "hpgd_sarsa": colors[1],
    "hpgd_oracle": colors[0],
    "sobirl": colors[5],
    "baseline": colors[6],
}
zorder=dict(zip(legend_names.keys(), [len(legend_names)-i for i in range(len(legend_names))]))

# --- Loading Data --- #

grid_search_outputs = {}
grid_search_params = {}
grid_search_train_states = {}
summary_dfs = {}


if save_figs and not os.path.exists(save_dir):
    os.makedirs(save_dir)

with open(os.path.join(data_dir, "config.yaml"), "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

print("Experimental Config:")
print(config)

rng = jax.random.PRNGKey(config["random_seed"])

# Environment Setup
env, env_params, incentive_train_state, config_incentive = environment_setup(rng, config)

found = []
for algo in algorithms:
    try:
        print(f"\n--- {algo} ---")
        with open(os.path.join(data_dir, f"metrics_{algo}.pkl"), "rb") as f:
            outputs = pickle.load(f)
        grid_search_outputs[algo] = outputs
        with open(os.path.join(data_dir, f"grid_search_{algo}.pkl"), "rb") as f:
            grid_params = pd.DataFrame(pickle.load(f))
        grid_search_params[algo] = grid_params
        print("Grid param keys: ", grid_search_params[algo].columns)
        for k in outputs:
            if jnp.any(jnp.isnan(outputs[k])):
                print(f"\033[91m[WARNING] Found NaN values in {k} for {algo}\033[0m")

        # Load orbax checkpoint
        ckpt_path = os.path.join(data_dir, f"checkpoint_incentive_{algo}")
        orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()
        grid_search_train_states[algo] = orbax_checkpointer.restore(ckpt_path)
        print("Incentive Train State Loaded")
    except:
        print(f"Failed to load {algo}")
        continue

    df = grid_params.copy()
    df["UL_initial_value"] = jnp.mean(outputs["UL_initial_value"][:, -1000:], axis=1)  # average across the last 1000 steps
    df = df.set_index(list(grid_params.keys()))
    df = df.groupby(list(grid_params.keys())).mean()  # average across seeds
    df = df.sort_values("UL_initial_value", ascending=False)
    display(df)
    summary_dfs[algo] = df
    found.append(algo)

algorithms = found

In [ ]:
# Merge results
combined_results = pd.DataFrame()

for algo in algorithms:
    if algo in summary_dfs:
        df = summary_dfs[algo].copy()
        df = df.reset_index()
        df['algorithm'] = algo
        combined_results = pd.concat([combined_results, df], ignore_index=True)

# Pivot by incentive_reg_grid for readability
if not combined_results.empty:
    pivot_table = combined_results.pivot_table(
        values='UL_initial_value',
        index=['incentive_reg_grid','lr_grid'], 
        columns='algorithm',
        aggfunc='first'
    )
    
    print("Combined Results Table")
    print("=" * 60)
    print(pivot_table.to_string())
    

# Performance Table

In [ ]:
# Find the best parameters for each reg_lambda
best_parameters_for_reg_lambda = {}
for algo in algorithms:
    if not algo in grid_search_params:
        print(f"Skipping {algo} as it is not in grid_search_params")
        continue
    df_grid_params = grid_search_params[algo]
    grouped = df_grid_params.groupby(list(df_grid_params.columns))
    df = summary_dfs[algo]
    best_params = {}
    for reg_lambda in jnp.unique(df_grid_params["incentive_reg_grid"].values):
        reg_lambda = float(reg_lambda)
        df_selected = df.loc[(slice(None), reg_lambda), :]
        best_params[reg_lambda] = df_selected.index[0]
        print(f"Best parameters for {algo} at reg_lambda {reg_lambda}: {best_params[reg_lambda]}")
    best_parameters_for_reg_lambda[algo] = best_params

# Create a summary table
df_summary_mean = pd.DataFrame()
df_summary_std_error = pd.DataFrame()
for reg_lambda in jnp.unique(grid_params["incentive_reg_grid"].values):
    reg_lambda = float(reg_lambda)
    for algo in algorithms:
        if not algo in grid_search_params:
            print(f"Skipping {algo} as it is not in grid_search_params")
            continue
        df_grid_params = grid_search_params[algo]
        outputs = grid_search_outputs[algo]
        idx = (df_grid_params == best_parameters_for_reg_lambda[algo][reg_lambda]).all(1).values
        UL_estimate = jnp.mean(outputs["UL_initial_value"][idx, -1000:], -1)
        df_summary_mean.loc[reg_lambda, algo] = jnp.mean(UL_estimate)  # Average across seeds
        df_summary_std_error.loc[reg_lambda, algo] = jnp.std(UL_estimate)/jnp.sqrt(UL_estimate.shape[0])  # Standard error across seeds
print("\nAverage Upper-Level Value (Last 1000 steps)")
display(df_summary_mean)
print("Std. Error Upper-Level Value (Last 1000 steps)")
display(df_summary_std_error)
print("Note: The row index is incentive_reg_grid, the column index is algorithm.")

# Visualize the learning curves

In [ ]:
for reg_lambda in jnp.unique(grid_params["incentive_reg_grid"].values):
    reg_lambda = float(reg_lambda)
    print(f"\n--- Reg_lambda: {reg_lambda} ---")
    input_data = {}
    max_outer_iterations = 0
    for algo in algorithms:
        if not algo in grid_search_params:
            print(f"Skipping {algo} as it is not in grid_search_params")
            continue
        df_grid_params = grid_search_params[algo]
        outputs = grid_search_outputs[algo]
        idx = (df_grid_params == best_parameters_for_reg_lambda[algo][reg_lambda]).all(1).values  # Shape: (n_grid_params,)
        input_data[algo] = outputs["UL_initial_value"][idx] if seed_idx is None else outputs["UL_initial_value"][idx][seed_idx].reshape(1,-1)  # Shape: (n_seeds, n_outer_iterations) or (1, n_outer_iterations)
        if input_data[algo].shape[1] > max_outer_iterations:
            max_outer_iterations = input_data[algo].shape[1]

    plot_UL_rewards(
        input_data,
        figsize=figsize,
        rolling_window=rolling_window,
        savefig_path=os.path.join(save_dir, f"UL_rewards_grid_search_reg_lambda_{reg_lambda}.pdf") if save_figs else None,
        xlim=(0, config["upper_optimisation"]["num_outer_iter"]),
        ylim=(-1.2, 1.6) if config["lower_optimisation"]["reg_lambda"] == 0.005 else (-1.2, 1.2),
        legend_position={
            "loc": "lower center",
            "bbox_to_anchor": (0.5, 0),
            "ncol": 2,
            "columnspacing": 0.7,
            "labelspacing": 0.35,
            "borderpad": 0.25,
        },
        legend_names=legend_names,
        line_styles=line_styles,
        algo_colors=algo_colors,
        zorder=zorder,
        yscale="linear"
    )
    plt.close()
plt.clf()

# Visualize the penalty maps

In [ ]:
seed_idx = 0 if seed_idx is None else seed_idx
from src.environments.ConfigurableFourRooms import map_project
for reg_lambda in jnp.unique(grid_params["incentive_reg_grid"].values):
    reg_lambda = float(reg_lambda)
    config_tmp = config.copy()
    print(f"\n--- Reg_lambda: {reg_lambda} ---")
    input_data = {}
    for algo in algorithms:
        df_grid_params = grid_search_params[algo]
        train_state = grid_search_train_states[algo]
        idx = (df_grid_params == best_parameters_for_reg_lambda[algo][reg_lambda]).all(1).values  # Shape: (n_grid_params,)
        input_data[algo] = jax.tree_map(lambda x: jnp.array(x)[idx], train_state["params"])

    fig, axes = plt.subplots(1, len(input_data), figsize=(21, 7), constrained_layout=True)

    # Ensure axes is always a list
    if len(input_data) == 1:
        axes = [axes]

    for i, (algo, incentive_train_state) in enumerate(input_data.items()):
        print(f"\n--- {algo} ---")
        axes[i].set_title(legend_names[algo])
        pcm = plot_incentive_grid(
            env,
            env_params,
            incentive_train_state["params"]["weights"][seed_idx],
            config_incentive["coordinates"],
            config_tmp,
            verbose=False,
            plot_input=(fig, axes[i]),
            cmap="PuRd_r",
        )

        # Add policy steps to the map
        init_probs = env.state_initialization_distribution(env_params.state_initialization_params).probs
        init_probs_mask = init_probs > 1e-8
        env_params_viz = env_params.replace(
            incentive_params=jax.tree_map(lambda x: x[seed_idx], incentive_train_state)
        )
        config_lower_level = config["lower_optimisation"]
        q_final, _ = general_value_iteration(
            env,
            env_params_viz,
            config_lower_level["discount_factor"],
            n_policy_iter=config_lower_level["n_policy_iter"],
            n_value_iter=config_lower_level["n_value_iter"],
            regularization=config_lower_level["regularization"],
            reg_lambda=config_lower_level["reg_lambda"],
            return_q_value=True,
        )
        br_policy = jax.nn.softmax(q_final/config_lower_level["reg_lambda"], axis=-1)  # Shape: (n_goals, n_states, n_actions)
        for j in range(len(env.available_goals)):
            goal_pos = jnp.array(config["environment"]["available_goals"][j])
            pos = env.available_init_pos[init_probs_mask][0]
            # Add the policy trajectories to the map
            for _ in range(30):
                try:
                    pos_idx = jnp.where(jnp.all(env.coords == pos[None, :], 1))[0][0]
                except:
                    print(pos)
                    break
                action_sort = jnp.argsort(br_policy[j, pos_idx])[::-1]
                for action in action_sort:  # Try actions in descending probability order and choose the first feasible move
                    action_direction = env.directions[action]
                    new_pos = map_project(env.env_map, pos, pos + action_direction)
                    if not jnp.all(new_pos == pos):
                        break
                color = "gray"
                axes[i].arrow(
                    pos[1],
                    pos[0],
                    action_direction[1]/2.0,
                    action_direction[0]/2.0,
                    head_width=0.1,
                    head_length=0.1,
                    linewidth=linewidth,
                    fc=color,
                    ec=color,
                    alpha=0.9,
                )
                pos = new_pos
                if jnp.all(pos == goal_pos):
                    break


        # Add goal and initial position annotations
        for j in range(len(env.available_goals)):
            goal_pos = config["environment"]["available_goals"][j]
            axes[i].annotate(
                rf"$\textbf{{G}}^{j+1}$",
                xy=(goal_pos[1], goal_pos[0]),
                xycoords="data",
                xytext=(goal_pos[1] - 0.3, goal_pos[0] + 0.25),
            )
        init_states_counter = 1
        for pos in env.available_init_pos[init_probs_mask]:
            axes[i].annotate(
                rf"$\textbf{{S^{init_states_counter}}}$" if sum(init_probs_mask) > 1 else rf"$\textbf{{S}}$",
                # fontsize=20,
                weight="bold",
                xy=(pos[1], pos[0]),
                xycoords="data",
                xytext=(pos[1]-0.3, pos[0]+0.25),

            )
            init_states_counter += 1
        pos = config["upper_optimisation"]["reward_function"]["target_state"]
        axes[i].annotate(
            rf"$\textbf{{+1}}$" if config["upper_optimisation"]["reward_function"]["type"] == "positive" else rf"$\textbf{{-1}}$",
            # fontsize=20,
            weight="bold",
            xy=(pos[1], pos[0]),
            xycoords="data",
            xytext=(pos[1]-0.3, pos[0]+0.25),
        )
        # Calculate the upper_level_value
        unused_pct = 100*jax.nn.softmax(incentive_train_state['params']['weights'][seed_idx])[-1]
        print(f"Unused percentage: {unused_pct:.4f}%")
    ax_list = axes if isinstance(axes, list) else axes.ravel().tolist() if hasattr(axes, 'ravel') else [axes]
    cbar = fig.colorbar(pcm, ax=ax_list, shrink=0.8)
    cbar.outline.set_visible(False)
    #plt.subplots_adjust(right=0.98)
    if save_figs:
        fig.savefig(os.path.join(save_dir, f"incentive_grid_grid_search_seed_{seed_idx}_reg_lambda_{reg_lambda}.pdf"), bbox_inches='tight')
    plt.show()
    plt.close()